# AE 370 Group 21: Atmospheric Entry Simulation Parameters
This notebook defines all physical constants, vehicle properties, and initial conditions (IVP) required to solve the equations of motion for a spacecraft returning from deep space.


In [8]:
from sympy import *
from scipy.integrate import solve_ivp
import numpy as np
import matplotlib.pyplot as plt

In [9]:
# --- Planetary Constants (Earth) ---
R_E = 6378137.0         # Radius of Earth (meters)
g0 = 9.80665            # Standard gravity at sea level (m/s^2)

# --- Atmospheric Model (Exponential approximation) ---
rho_0 = 1.225           # Sea-level atmospheric density (kg/m^3)
H_scale = 7500.0        # Atmospheric scale height (meters)

def get_density(h):
    """Calculates atmospheric density at a given altitude using an exponential model."""
    if h < 0:
        return rho_0
    return rho_0 * np.exp(-h / H_scale)

def get_gravity(h):
    """Calculates gravity acceleration variation with altitude."""
    return g0 * (R_E / (R_E + h))**2


In [10]:
# --- Vehicle Characteristics ---
mass = 5000.0           # Spacecraft mass (kg)
area = 10.0             # Reference frontal area (m^2)

# --- Aerodynamic Coefficients ---
# Note: These can be fixed constants or updated as a function of alpha (angle of attack)
C_D = 1.5               # Drag coefficient
C_L = 0.5               # Lift coefficient


In [11]:
# --- Initial Conditions at Entry Interface (t = 0) ---
h_0 = 120000.0          # Initial entry altitude (meters, typically 120 km)
v_0 = 11200.0           # Initial entry speed from deep space (m/s, ~escape velocity)
gamma_0 = np.radians(-6.0) # Initial flight-path angle (converted from degrees to radians)

# Collect into an initial state vector for scipy.integrate.solve_ivp
# State layout: [altitude, velocity, flight-path angle]
initial_state = [h_0, v_0, gamma_0]


## Reusable Functions for Main Simulation
These helper functions compute Lift, Drag, and total g-load ($n$) to easily compute tracking variables at each timestamp.


In [12]:
def calc_aerodynamics(h, v, C_L=C_L, C_D=C_D, mass=mass, area=area):
    """Computes lift and drag force magnitudes along with the total g-load."""
    rho = get_density(h)
    dynamic_pressure = 0.5 * rho * (v**2)
    
    lift = dynamic_pressure * C_L * area
    drag = dynamic_pressure * C_D * area
    
    # Calculate load factor in g's: n = sqrt(L^2 + D^2) / (mass * g0)
    g_load = np.sqrt(lift**2 + drag**2) / (mass * g0)
    
    return lift, drag, g_load
